# Accuracy-Improvement Workstreams on the Diploma Pipeline

**AITU, June 2026 — Yerassyl Raimkhan**

This notebook is the second extension of `diploma_pipeline.ipynb`. It builds on
the 111-feature weather model (`weather_extension.ipynb`) with three new levers,
applied sequentially:

1. **Feature bundle** — French national holidays + Paris Zone-C school holidays,
   day-of-year cyclical, HDD/CDD (heating/cooling degree days, base 18.3 °C),
   plus temperature-by-daypart and `power_lag_1h × temp_now` interactions.
   Result: **125 features** (111 + 14).
2. **Optuna retune** — 60 trials on the same 9-parameter search space as
   Phase 5, applied to the enlarged feature set.
3. **Stacking ensemble** — Constrained-LSQ blend (or RidgeCV, picked by smaller
   val→test R² gap) over five base models:
   `xgb69`, `xgb111`, `xgb125_tuned`, `lstm69`, `gru69`.

**Constraints inherited from earlier work:**
- Identical chronological split boundaries; identical 5,075-hour aligned test
  window; `MAPE_EPSILON_KW = 0.10`; `SEED = 42` throughout.
- All writes scoped to `outputs/improvements/`. Original `outputs/` and
  `outputs/weather_extension/` are read-only.

The heavy logic lives in `accuracy_improvements.py` — each notebook cell here
invokes one of its `stepN_*` functions and renders the outputs inline. Re-runs
are cached by parquet/npy/joblib so the notebook is fast after the first pass.


## Section 0 — Setup

Imports the step functions from the companion script and seeds RNGs. The
script's section functions are idempotent: each one looks for its cached
artifact under `outputs/improvements/` and skips computation if present.


In [1]:
from __future__ import annotations
import os, random, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)

import accuracy_improvements as ai
print("script module loaded from", ai.__file__)
print("OUT directory:", ai.OUT)


script module loaded from C:\Users\ЕРАСЫЛ\Desktop\diplom code\accuracy_improvements.py
OUT directory: C:\Users\ЕРАСЫЛ\Desktop\diplom code\outputs\improvements


## Section 1 — Feature bundle (111 → 125)

**Calendar (5):** `is_holiday_fr` (FR national holidays via the `holidays`
package), `is_school_holiday_fr_c` (Paris Zone-C school vacation windows from
a static lookup table), `is_long_weekend` (pont days adjacent to Tuesday/
Thursday holidays), `doy_sin`/`doy_cos` (day-of-year cyclical — captures
Christmas/August vacation dips that monthly encodings blur).

**HDD/CDD (3):** `hdd = max(0, 18.3 − T)`, `cdd = max(0, T − 18.3)` (ASHRAE
residential base), and `hdd_rolling_24h` (`.shift(1).rolling(24).mean()` for
leakage-safe multi-day cold-snap memory).

**Interactions (6):** `temp_x_hour_{sin,cos}`, `temp_x_is_weekend`,
`radiation_x_hour_{sin,cos}`, and `power_lag_1h_x_temp_now` (lagged household
demand conditioned on current temperature — the strongest of the six in SHAP).


In [2]:
features = ai.step1_features()
features.shape, list(features.columns)[-14:]   # last 14 = the new bundle


[step1] cache hit: C:\Users\ЕРАСЫЛ\Desktop\diplom code\outputs\improvements\features_125.parquet
[step1] shape=(33965, 126)  features=125
[step1] new feature summary:
                              mean        std        min        max
is_holiday_fr             0.029678   0.169699   0.000000   1.000000
is_school_holiday_fr_c    0.339025   0.473385   0.000000   1.000000
is_long_weekend           0.010187   0.100417   0.000000   1.000000
hdd                       7.642850   6.146321   0.000000  28.799500
cdd                       0.514988   1.600993   0.000000  16.450500
hdd_rolling_24h           7.642633   5.703777   0.000000  25.518249
temp_x_hour_sin          -1.349454   9.146995 -34.250500  22.651444
power_lag_1h_x_temp_now  11.021412  11.437019 -24.345891  94.924949


((33965, 126),
 ['is_holiday_fr',
  'is_school_holiday_fr_c',
  'is_long_weekend',
  'doy_sin',
  'doy_cos',
  'hdd',
  'cdd',
  'hdd_rolling_24h',
  'temp_x_hour_sin',
  'temp_x_hour_cos',
  'temp_x_is_weekend',
  'radiation_x_hour_sin',
  'radiation_x_hour_cos',
  'power_lag_1h_x_temp_now'])

## Section 2 — Chronological split + StandardScaler

Boundaries inherited from `weather_extension/splits/*` so the test window
remains the same 5,099 contiguous hours (April 2010 → November 2010) that the
weather model was evaluated on. Scaler fit on the 125 train features only.


In [3]:
train, val, test, scaler = ai.step2_split(features)
print("scaler.n_features_in_ =", scaler.n_features_in_)


[step2] cache hit: splits + scaler


[step2] train=23768 val=5098 test=5099
[step2] train range: 2006-12-24 17:00:00+01:00 -> 2009-09-15 18:00:00+01:00
[step2] val   range: 2009-09-15 19:00:00+01:00 -> 2010-04-19 18:00:00+01:00
[step2] test  range: 2010-04-19 19:00:00+01:00 -> 2010-11-26 20:00:00+01:00
scaler.n_features_in_ = 125


## Section 3 — Optuna retune on the 125-feature set

Same 9-parameter search space as `diploma_pipeline.ipynb` Cells 33–37, but
**60 trials** instead of 40 and applied to the enlarged feature set. The TPE
sampler is seeded; saved best parameters in `outputs/improvements/
optuna_125_best_params.json`. The final model is refit with the best params
and produces both val and test predictions (the val predictions feed into
the stacking meta-learner in Section 5).

> **First run takes ~5–10 minutes**; subsequent runs hit the cache and are
> instant.


In [4]:
tuned, X_test, y_test, pred_test_125, pred_val_125, metrics_125, feature_cols = \
    ai.step3_optuna(train, val, test, scaler)
print("aligned test metrics:", metrics_125)


[step3] cache hit: study + model + predictions


[step3] aligned val:  {'RMSE': 0.4973991718490001, 'MAE': 0.3406108814571295, 'R2': 0.6884453727464047, 'MAPE': 35.230186459854465}
[step3] aligned test: {'RMSE': 0.434910116545744, 'MAE': 0.2990101587556865, 'R2': 0.6247188872996781, 'MAPE': 39.006086136249095}
[step3] vs weather (R^2 0.6187): DeltaR^2 = +0.0060
aligned test metrics: {'RMSE': 0.434910116545744, 'MAE': 0.2990101587556865, 'R2': 0.6247188872996781, 'MAPE': 39.006086136249095}


## Section 4 — Base-model predictions for stacking

For each of the five stacking base models, produce aligned val + test
predictions (both 5,074- and 5,075-row windows respectively). Val
predictions for the tree models are regenerated from the saved models;
LSTM/GRU val predictions are produced by sliding 24-step sequences from
the last 24 train rows through the val set. All test predictions are the
already-aligned arrays from the original pipeline.


In [5]:
base, y_val_aligned, y_test_aligned = ai.step4_base_predictions()
print("base models loaded; aligned shapes:")
for name in ["xgb69", "xgb111", "xgb125", "lstm", "gru"]:
    print(f"  {name:6s}  val={base[name]['val'].shape}  test={base[name]['test'].shape}")


[step4] base-model aligned metrics:
  model         val RMSE   val R^2   test RMSE  test R^2
  xgb69           0.7364    0.3172      0.4402    0.6156
  xgb111          0.5055    0.6782      0.4384    0.6187
  xgb125          0.4974    0.6884      0.4349    0.6247
  lstm            0.6419    0.4812      0.5586    0.3809
  gru             0.6323    0.4966      0.5627    0.3718
base models loaded; aligned shapes:
  xgb69   val=(5074,)  test=(5075,)
  xgb111  val=(5074,)  test=(5075,)
  xgb125  val=(5074,)  test=(5075,)
  lstm    val=(5074,)  test=(5075,)
  gru     val=(5074,)  test=(5075,)


## Section 5 — Stacking ensemble

Two meta-learner variants are fit on the **val** predictions and chosen by
the smaller val→test R² gap (a proxy for generalisation):

- **Constrained-LSQ blend** — weights ≥ 0 summing to 1 (SLSQP), interpretable
  as model votes.
- **RidgeCV stacking** — `RidgeCV(alphas=np.logspace(-3, 3, 20), cv=5)`,
  unconstrained linear meta with regularised coefficients.

The chosen variant is saved as `outputs/improvements/blend_weights.json` plus
(if Ridge) `ridge_meta.joblib`. The final test prediction is clipped to
[0, 20] kW and written to `stacked_predictions.npy`.


In [6]:
metrics_stack, chosen, blend_info = ai.step5_stacking(
    base, y_val_aligned, y_test_aligned)
print(f"chosen variant: {chosen}")
print(f"stacked test:   {metrics_stack}")


[step5] blend weights: {'xgb69': 0.0, 'xgb111': 0.152, 'xgb125': 0.823, 'lstm': 0.0, 'gru': 0.025}
[step5] ridge coefs:   {'xgb69': -0.129, 'xgb111': 0.172, 'xgb125': 0.85, 'lstm': -0.039, 'gru': 0.061}  intercept=0.0957
[step5] val->test R^2 gap: blend=0.0626  ridge=0.0673  -> chosen=blend
[step5] stacked  val:  {'RMSE': 0.4970120905005224, 'MAE': 0.3403375553513254, 'R2': 0.6889300943427943, 'MAPE': 35.26943025701354}
[step5] stacked  test: {'RMSE': 0.43396962617419155, 'MAE': 0.2984731334703746, 'R2': 0.6263402184000819, 'MAPE': 38.95832542435344}
chosen variant: blend
stacked test:   {'RMSE': 0.43396962617419155, 'MAE': 0.2984731334703746, 'R2': 0.6263402184000819, 'MAPE': 38.95832542435344}


## Section 6 — Report, SHAP, comparison plot, bootstrap CI

The four-row model-progression table:

| Stage             | Features | Note                              |
|-------------------|----------|-----------------------------------|
| Baseline          | 69       | Original `diploma_pipeline.ipynb` |
| +Weather          | 111      | `weather_extension.ipynb`         |
| +Cal/HDD/Inter+Optuna | 125  | This notebook, Sections 1–3       |
| Stacked ensemble  | 5-base   | This notebook, Sections 4–5       |

SHAP is computed via `TreeExplainer` on 500 random test rows (seeded), with
new colour categories for `calendar`, `hdd`, and `interaction` features. The
bootstrap CI on Δ R² (stacked − baseline) uses 1,000 resamples of the 5,075-
hour test window — if the 95% CI excludes zero, the improvement is
statistically distinguishable from sampling noise.


In [7]:
cmp, ranking, ci = ai.step6_report(
    tuned, test, scaler, feature_cols, metrics_125, metrics_stack,
    base, y_test_aligned, chosen, blend_info)
print()
print((ai.OUT / "improvement_report.txt").read_text(encoding="utf-8"))


[step6] comparison saved -> C:\Users\ЕРАСЫЛ\Desktop\diplom code\outputs\improvements\comparison_table.csv


[step6] bootstrap Delta R^2 (stacked - xgb69): mean=+0.0106  95% CI=[+0.0042, +0.0170]
[step6] wrote report -> C:\Users\ЕРАСЫЛ\Desktop\diplom code\outputs\improvements\improvement_report.txt

=== ACCURACY IMPROVEMENTS REPORT ===

Dataset:    UCI IHEPC, Sceaux, France, hourly (2006-12 -> 2010-11)
Test win.:  aligned 5,075 hours (test[24:])
Pipeline:   feature bundle -> Optuna(60 trials) -> stacking

--- Model progression ---

Model                                       Features      RMSE       MAE        R^2       MAPE     Delta R^2
------------------------------------------- --------    ---------  ---------  --------  ---------  ---------
XGBoost (69 feat, baseline)                       69       0.4402     0.3008    0.6156    39.3600    +0.0000
XGBoost (111 feat, +weather)                     111       0.4384     0.2995    0.6187    39.0000    +0.0031
XGBoost (125 feat, +cal+HDD+inter, Optuna)       125       0.4349     0.2990    0.6247    39.0061    +0.0091
Stacked ensemble (blend)  

## Final artifact check

Verifies every required artifact under `outputs/improvements/` is present
and non-empty.


In [8]:
from pathlib import Path
expected = [
    "features_125.parquet",
    "splits/train_125.parquet", "splits/val_125.parquet", "splits/test_125.parquet",
    "scaler_125.joblib",
    "optuna_125_study.pkl", "optuna_125_trials.csv", "optuna_125_best_params.json",
    "xgb_125_tuned.json", "pred_xgb_125_tuned.npy", "pred_xgb_125_tuned_val.npy",
    "preds_base/xgb69_val.npy", "preds_base/xgb69_test.npy",
    "preds_base/xgb111_val.npy", "preds_base/xgb111_test.npy",
    "preds_base/xgb125_val.npy", "preds_base/xgb125_test.npy",
    "preds_base/lstm_val.npy", "preds_base/lstm_test.npy",
    "preds_base/gru_val.npy", "preds_base/gru_test.npy",
    "stacked_predictions.npy", "blend_weights.json",
    "shap_125_features.json", "shap_125_bar.png",
    "comparison_table.csv", "model_comparison.png", "improvement_report.txt",
]
missing = [p for p in expected if not (ai.OUT / p).exists()
            or (ai.OUT / p).stat().st_size == 0]
assert not missing, f"missing or empty: {missing}"
print(f"all {len(expected)} artifacts verified")


all 28 artifacts verified
